In [1]:
import os
import json
import re
from pathlib import Path
import pdfplumber
import pytesseract
import cv2
import numpy as np
from PIL import Image
from dotenv import load_dotenv
from groq import Groq

# Load environment variables
load_dotenv()
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# Configure Tesseract path from environment (with fallback)
TESSERACT_PATH = os.getenv("TESSERACT_PATH", r"C:\Program Files\Tesseract-OCR\tesseract.exe")
pytesseract.pytesseract.tesseract_cmd = TESSERACT_PATH

# -----------------------------------------------------------------------------
# TEXT EXTRACTION UTILITIES
# -----------------------------------------------------------------------------

def extract_from_pdf(file_path):
    """Extract text from PDF documents"""
    text_content = []
    try:
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text_content.append(page_text)
        return "\n".join(text_content)
    except Exception as e:
        print(f"PDF extraction error: {e}")
        return ""

def extract_from_image(file_path):
    """Extract text from images using OCR with preprocessing"""
    try:
        # Read image and convert to grayscale
        img = cv2.imread(file_path)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        # Apply thresholding to improve OCR accuracy
        _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)
        
        # Run Tesseract
        text = pytesseract.image_to_string(thresh)
        return text
    except Exception as e:
        print(f"Image OCR error: {e}")
        return ""

def read_text_file(file_path):
    """Read plain text files"""
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            return f.read()
    except Exception as e:
        print(f"Text file error: {e}")
        return ""

def get_document_text(file_path):
    """Route to appropriate extractor based on file extension"""
    ext = Path(file_path).suffix.lower()
    
    if ext == ".pdf":
        return extract_from_pdf(file_path)
    elif ext in (".png", ".jpg", ".jpeg", ".jfif"):
        return extract_from_image(file_path)
    elif ext == ".txt":
        return read_text_file(file_path)
    else:
        return ""

# -----------------------------------------------------------------------------
# EXTRACTION PROMPT (as provided, with minor correction)
# -----------------------------------------------------------------------------

EXTRACTION_PROMPT = """You are a medical document information extraction system.

You are given the FULL OCR text of a diagnostic lab report.
Extract structured information and return ONLY valid JSON in the exact schema below.

====================
REQUIRED OUTPUT SCHEMA
====================
{
  "patient_information": {
    "patient_name": string | null,
    "age_years": number | null,
    "gender": string | null,
    "lab_number": string | null,
    "report_status": string | null,
    "collection_datetime": string | null,
    "report_datetime": string | null,
    "laboratory_name": string | null
  },

  "tests_index": {
    "<canonical_test_key>": {
      "test_name": string,
      "value": number | string | null,
      "units": string | null,
      "reference_range": string | null,
      "interpretation": "low" | "normal" | "high" | "borderline" | null,
      "category": string
    }
  },

  "tests_by_category": {
    "complete_blood_count": [ "<canonical_test_key>" ],
    "liver_function": [ "<canonical_test_key>" ],
    "kidney_function": [ "<canonical_test_key>" ],
    "lipid_profile": [ "<canonical_test_key>" ],
    "thyroid_profile": [ "<canonical_test_key>" ],
    "diabetes_related": [ "<canonical_test_key>" ],
    "vitamins_and_minerals": [ "<canonical_test_key>" ],
    "electrolytes": [ "<canonical_test_key>" ],
    "other_tests": [ "<canonical_test_key>" ]
  },

  "abnormal_findings": [
    {
      "canonical_test_key": string,
      "observed_value": number | string,
      "expected_range": string,
      "severity": "low" | "high" | "critical"
    }
  ],

  "clinical_notes": {
    "interpretations": [ string ],
    "comments": [ string ],
    "notes": [ string ],
    "recommendations": [ string ],
    "disclaimers": [ string ]
  },

  "metadata": {
    "total_pages": number,
    "page_numbers_detected": [ number ],
    "report_complete": boolean
  }
}

====================
CANONICAL TEST KEY RULES
====================
- Each test MUST appear exactly ONCE in tests_index
- Use normalized snake_case keys
- Remove symbols, punctuation, and methods
- Use same canonical key everywhere

====================
INTERPRETATION RULES
====================
- Use reference range to detect low/normal/high
- If borderline mentioned → borderline
- Otherwise → normal

====================
STRICT RULES
====================
- Return ONLY valid JSON
- No explanation
- No markdown
- No comments
- No duplicate tests
- Missing values → null
"""

# -----------------------------------------------------------------------------
# JSON CLEANING AND PARSING
# -----------------------------------------------------------------------------

def clean_json_response(raw_text):
    """Extract and clean JSON from LLM response"""
    # Remove markdown code fences if present
    cleaned = re.sub(r"```json\s*|\s*```", "", raw_text).strip()
    
    # Locate JSON object boundaries
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("No JSON object found in response")
    
    json_str = cleaned[start:end+1]
    
    # Fix trailing commas (common LLM issue)
    json_str = re.sub(r",\s*}", "}", json_str)
    json_str = re.sub(r",\s*]", "]", json_str)
    
    return json.loads(json_str)

def save_extracted_json(data, filename="output.json"):
    """Save extracted data to JSON file in output directory"""
    output_dir = Path("output")
    output_dir.mkdir(exist_ok=True)
    
    output_path = output_dir / filename
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    
    return str(output_path)

# -----------------------------------------------------------------------------
# LLM INTERACTION
# -----------------------------------------------------------------------------

def extract_medical_json(ocr_text):
    """Send OCR text to Groq LLM and return structured JSON"""
    full_prompt = EXTRACTION_PROMPT + "\n\nOCR TEXT:\n" + ocr_text
    
    try:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": "Return ONLY valid JSON. No text."},
                {"role": "user", "content": full_prompt}
            ],
            temperature=0.0,
            max_tokens=8000
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Groq API error: {e}")
        return "{}"

# -----------------------------------------------------------------------------
# MAIN PROCESSING FUNCTION
# -----------------------------------------------------------------------------

def process_medical_report(file_path, patient_name=None):
    """Main pipeline: extract text, call LLM, clean JSON, save output"""
    
    # Step 1: Extract raw text from document
    print(f"Extracting text from: {file_path}")
    ocr_text = get_document_text(file_path)
    
    if not ocr_text.strip():
        return {"error": "No text could be extracted from the document."}
    
    print(f"Extracted {len(ocr_text)} characters")
    
    # Step 2: Call LLM to structure the data
    print("Sending to LLM for structuring...")
    llm_output = extract_medical_json(ocr_text)
    
    # Step 3: Clean and parse JSON
    try:
        structured_data = clean_json_response(llm_output)
    except (json.JSONDecodeError, ValueError) as e:
        return {
            "error": "Failed to parse LLM response as JSON",
            "raw_output": llm_output,
            "exception": str(e)
        }
    
    # Optionally inject patient name if provided
    if patient_name and "patient_information" in structured_data:
        structured_data["patient_information"]["patient_name"] = patient_name
    
    # Step 4: Save to file
    saved_path = save_extracted_json(structured_data)
    structured_data["_metadata"] = {"saved_to": saved_path}
    
    return structured_data

# -----------------------------------------------------------------------------
# CLI ENTRY POINT
# -----------------------------------------------------------------------------

if __name__ == "__main__":
    import argparse
    
    parser = argparse.ArgumentParser(description="Extract structured medical data from reports")
    parser.add_argument("file", help="Path to medical report (PDF, image, or text)")
    parser.add_argument("--name", "-n", help="Patient name (overrides extracted)")
    parser.add_argument("--output", "-o", default="output.json", help="Output filename")
    
    args = parser.parse_args()
    
    result = process_medical_report(args.file, args.name)
    
    if "error" in result:
        print(f"Error: {result['error']}")
        if "raw_output" in result:
            print("\nRaw LLM output:\n", result["raw_output"])
    else:
        print(json.dumps(result, indent=2))
        print(f"\n✅ Structured data saved to output/{args.output}")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\HP\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\HP\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\HP\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\Users\HP\anaconda3\Lib\site-packages\tornado

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\HP\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\HP\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\HP\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\Users\HP\anaconda3\Lib\site-packages\tornado

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: numpy.core.multiarray failed to import